# 01 | HARTH Data Loading and Inspection

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Reuse our common `load_data`, `inspect_data` and `plot_signal` functions with a different biomedical movement dataset.

**Dataset:** Human Activity Recognition Trondheim, authors' official GitHub **v2.0** release. The first demonstration downloads only subject `S023.csv`. UCI has an older **v1.2** release of 22 participants; do not mix the versions. No model training is performed.

## 01. Imports and environment

Reuse the same Python functions as the BoneNet notebook. The HARTH-specific adapter is the only new data interpreter.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook from inside human-movement-intelligence.")
    root = root.parent

sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))

from core.data_loading import load_data
from core.data_inspection import inspect_data, plot_signal
from harth_data import (
    ACTIVITY_NAMES, SENSOR_COLUMNS,
    ensure_harth_sample, load_harth_subject,
)

print("Repository:", root)
print("Reusable loading and visualisation functions: ready")

## 02. Download only the selected public participant

The `S023.csv` source is cached in ignored `data/raw/harth_v2/`. It comes from an exact pinned revision of the authors' repository.

In [ ]:
data_dir = root / "data" / "raw" / "harth_v2"
csv_path = ensure_harth_sample(data_dir, subject="S023")
print("Source file:", csv_path)

## 03. General file reading and structural inspection

First call the same general `load_data()` function as in the BoneNet notebook. It does not encode HARTH-specific scientific meaning.

In [ ]:
raw_table = load_data(csv_path)
raw_report = inspect_data(raw_table, name="HARTH raw participant CSV")
print("Raw columns:", list(raw_table.columns))

## 04. Validate this source with the HARTH adapter

Validate timestamp parsing, six acceleration columns in **g**, and activity codes. This remains one continuous recording from one person.

In [ ]:
data, channel_names = load_harth_subject(csv_path)
summary = inspect_data(data, name="HARTH validated participant")
print("Subject identifier:", csv_path.stem)
print("Accelerometer channels:", channel_names)
print("Sampling rate (source documentation): 50 Hz")

## 05. Explore the true label distribution

Activity codes are published by the authors. Counts in this cell are calculated from the **actual** `S023.csv` data.

In [ ]:
activity_counts = data["label"].value_counts().sort_index()
activity_table = pd.DataFrame({
    "label": activity_counts.index,
    "activity": [ACTIVITY_NAMES[int(value)] for value in activity_counts.index],
    "samples": activity_counts.to_numpy(),
})
display(activity_table)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(activity_table["activity"], activity_table["samples"])
ax.set(title="HARTH S023: annotated activity sample counts",
       xlabel="Activity", ylabel="Recorded samples")
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()

## 06. Plot original movement signals

These are observed sensor values for the first 250 samples; this is not a machine-learning prediction. The units are **g**, as documented by HARTH.

In [ ]:
n_points = min(250, len(data))
signal = data.loc[:n_points - 1, channel_names[:3]].to_numpy().T
fig = plot_signal(
    signal,
    title="HARTH S023: lower-back accelerometer (first 250 samples)",
    channel_names=channel_names[:3],
)
fig.axes[0].set_ylabel("Acceleration (g)")
plt.show()

## 07. Data-quality gate and takeaways

This shows successful reuse across source formats. A single subject cannot support unseen-subject generalisation experiments. Windowing, participant-disjoint splits and model training belong in later notebooks.

In [ ]:
assert len(channel_names) == 6
assert set(["timestamp", "label"]).issubset(data.columns)
assert data[channel_names].notna().all().all()
assert not data.empty
time_delta = data["timestamp"].diff().dt.total_seconds().dropna()
print("Number of rows:", len(data))
print("Median observed time interval (s):", time_delta.median())
print("Observed activity codes:", sorted(data["label"].unique().tolist()))
print("Generic reader and inspector reused: YES")
print("Model training performed: NO")